# kaggle-vllm 0.2.0 fresh acceptance

Status: **pending execution on Kaggle T4 x2**. This notebook intentionally has no saved outputs. Before a final `0.2.0` release exists, it accepts either an explicitly attached reviewed `0.2.0.dev0` SDK wheel or builds the small SDK wheel from the exact reviewed GitHub source commit. Historical notebooks remain unchanged.


In [ ]:
from pathlib import Path
import hashlib
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request

# Reviewed 0.2 development source: PR #12 merge.
EXPECTED_SOURCE_COMMIT = "8357731845d8b36b5793fcf394045ab639d0c002"
EXPECTED_SDK_VERSION = "0.2.0.dev0"

# Optional attached candidate built locally from the reviewed source.
# This is NOT a published 0.2.0 release asset.
EXPECTED_ATTACHED_WHEEL_NAME = "kaggle_vllm-0.2.0.dev0-py3-none-any.whl"
EXPECTED_ATTACHED_WHEEL_SHA256 = "ba50a7617ef18830424ad07b58eefc2dbcb479d9bf60a2f5da0fe3ba42cd4216"

INPUT_ROOT = Path("/kaggle/input")
SDK_WORK_ROOT = Path("/kaggle/working/kaggle-vllm-sdk-020dev0")
SDK_DIST = SDK_WORK_ROOT / "dist"
SOURCE_ARCHIVE = SDK_WORK_ROOT / f"kaggle-vllm-{EXPECTED_SOURCE_COMMIT}.tar.gz"
SOURCE_URL = (
    "https://github.com/kaggle-vllm/kaggle-vllm/archive/"
    f"{EXPECTED_SOURCE_COMMIT}.tar.gz"
)

def sha256_file(path: Path, chunk_size: int = 8 * 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()

def safe_extract_tar(archive: Path, destination: Path) -> None:
    destination = destination.resolve()
    with tarfile.open(archive, "r:gz") as tf:
        for member in tf.getmembers():
            if member.issym() or member.islnk():
                raise RuntimeError(f"Refusing archive link member: {member.name}")
            target = (destination / member.name).resolve()
            if target != destination and destination not in target.parents:
                raise RuntimeError(f"Unsafe archive member: {member.name}")
        tf.extractall(destination)

# Record Torch identity before installing either build tooling or the SDK.
torch_before = subprocess.check_output(
    [
        sys.executable,
        "-c",
        "import torch; print(torch.__version__, torch.version.cuda, torch.__file__)",
    ],
    text=True,
)

# Preferred path: use the exact attached candidate if it is present.
all_wheels = sorted(INPUT_ROOT.rglob("*.whl")) if INPUT_ROOT.exists() else []
attached = [p for p in all_wheels if p.name == EXPECTED_ATTACHED_WHEEL_NAME]

if attached:
    if len(attached) != 1:
        raise RuntimeError(
            "Expected exactly one attached 0.2.0.dev0 SDK wheel, found:\n"
            + "\n".join(f"  - {p}" for p in attached)
        )
    SDK_WHEEL = attached[0]
    sdk_sha256 = sha256_file(SDK_WHEEL)
    if sdk_sha256 != EXPECTED_ATTACHED_WHEEL_SHA256:
        raise RuntimeError(
            "Attached SDK wheel SHA256 mismatch.\n"
            f"Expected: {EXPECTED_ATTACHED_WHEEL_SHA256}\n"
            f"Actual:   {sdk_sha256}\n"
            f"Path:     {SDK_WHEEL}"
        )
    SDK_SOURCE_MODE = "attached-reviewed-wheel"
else:
    # No 0.2.0/dev0 release asset is required before release acceptance.
    # Build the lightweight SDK from the exact reviewed source commit instead.
    if SDK_WORK_ROOT.exists():
        shutil.rmtree(SDK_WORK_ROOT)
    SDK_DIST.mkdir(parents=True, exist_ok=True)

    print("No attached 0.2.0.dev0 SDK wheel found.")
    print("Building the SDK from reviewed GitHub commit:", EXPECTED_SOURCE_COMMIT)

    urllib.request.urlretrieve(SOURCE_URL, SOURCE_ARCHIVE)
    source_unpack = SDK_WORK_ROOT / "source"
    source_unpack.mkdir(parents=True, exist_ok=True)
    safe_extract_tar(SOURCE_ARCHIVE, source_unpack)

    roots = [p for p in source_unpack.iterdir() if p.is_dir()]
    if len(roots) != 1:
        raise RuntimeError(f"Unexpected GitHub source archive layout: {roots}")
    source_root = roots[0]

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--no-cache-dir", "build>=1.2"],
        check=True,
    )
    build_env = os.environ.copy()
    build_env["SOURCE_DATE_EPOCH"] = "1787760000"
    subprocess.run(
        [
            sys.executable,
            "-m",
            "build",
            "--wheel",
            "--outdir",
            str(SDK_DIST),
            str(source_root),
        ],
        check=True,
        env=build_env,
    )

    built = sorted(SDK_DIST.glob("kaggle_vllm-0.2.0.dev0-py3-none-any.whl"))
    if len(built) != 1:
        raise RuntimeError(f"Expected one built 0.2.0.dev0 wheel, found: {built}")

    SDK_WHEEL = built[0]
    sdk_sha256 = sha256_file(SDK_WHEEL)
    SDK_SOURCE_MODE = "built-from-reviewed-github-commit"

print("SDK source mode:", SDK_SOURCE_MODE)
print("SDK wheel:", SDK_WHEEL)
print("SDK SHA256:", sdk_sha256)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "--no-cache-dir", str(SDK_WHEEL)],
    check=True,
)

import kaggle_vllm

assert kaggle_vllm.__version__ == EXPECTED_SDK_VERSION, (
    kaggle_vllm.__version__,
    EXPECTED_SDK_VERSION,
)
print("SDK version:", kaggle_vllm.__version__)
print("SDK candidate identity: PASS")


In [ ]:
# Use notebook-owned runtime paths so rerunning the acceptance does not collide
# with an old default staged/overlay directory.
RUNTIME_ROOT = Path("/kaggle/working/kaggle-vllm-e2e-020dev0")
STAGED = RUNTIME_ROOT / "vllm-staged"
OVERLAY = RUNTIME_ROOT / "vllm-runtime-overlay"
MANIFEST = RUNTIME_ROOT / "kaggle-vllm-runtime.json"
CACHE = Path("/kaggle/working/kaggle-vllm-cache")

if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True, exist_ok=True)
CACHE.mkdir(parents=True, exist_ok=True)

BOOTSTRAP = [
    "kaggle-vllm",
    "bootstrap",
    "--strict",
    "--staged", str(STAGED),
    "--overlay", str(OVERLAY),
    "--cache", str(CACHE),
    "--manifest", str(MANIFEST),
]

subprocess.run(["nvidia-smi"], check=True)
subprocess.run(["nvidia-smi", "topo", "-m"], check=True)
subprocess.run(["nvcc", "--version"], check=True)
subprocess.run(["kaggle-vllm", "fingerprint"], check=True)
subprocess.run(BOOTSTRAP + ["--dry-run", "--json"], check=True)
subprocess.run(BOOTSTRAP, check=True)

assert MANIFEST.is_file(), f"Runtime manifest missing: {MANIFEST}"
print("Native bootstrap: PASS")


In [ ]:
from kaggle_vllm import activate_runtime

assert activate_runtime(MANIFEST), f"Could not activate runtime manifest: {MANIFEST}"

import torch
import vllm
import vllm._C
import vllm._moe_C
import vllm.cumem_allocator

torch_after = f"{torch.__version__} {torch.version.cuda} {torch.__file__}\n"
assert torch_after == torch_before, (
    "Kaggle Torch identity changed",
    torch_before,
    torch_after,
)

doctor = subprocess.run(["kaggle-vllm", "doctor", "--strict", "--json"])
assert doctor.returncode == 0, f"doctor --strict failed: {doctor.returncode}"

print("Native imports: PASS")
print("Torch preservation: PASS")
print("Strict doctor: PASS")


Continue with `scripts/nccl_smoke.py`, OPT-125M TP=1 and TP=2 generation, optional Qwen `inspect-shards --tensor-parallel-size 2` and reload, local OpenAI endpoint checks, then the benchmark notebook. Preserve all outputs as new 0.2.0 evidence; do not overwrite historical notebooks.